# C2.9 · Case study — Moltbook: 770,000 agents behind one missing policy

**Function C — Red Teaming and Security Research with AI → Security Research with AI**  ·  *Both directions*

Builds on **[C2.8 · Case study — the Hugging Face / OpenAI agent-swarm incident](https://spbreed.github.io/cyber-commons/lessons/C2.8.html)**.

| | |
|---|---|
| Tools used | Supabase, PostgREST |

## What this lesson is

**What it covers.** Run the same query with and without a row policy, then work out which of the leaked things the platform could actually revoke.

**Why a security engineer needs it.** The blast radius was not the platform's. What leaked were credentials in five other providers' accounts, and the platform could revoke none of them. The control it builds is: row-level policies, credentials out of client-readable tables, and an admin plane the client cannot reach — the controls of A3.8, arriving at a database.

This is a **control** lesson: it builds the mechanism, then breaks it, so you can see what the control is actually load-bearing for rather than taking the claim on trust.

## 1 · The hook

A social network whose members were AI agents shipped its database key to every browser, which is normal, and left row-level security off, which is not. One query returned every agent's record — including the OpenAI, Anthropic and AWS keys of the people who created them, in plaintext.

> **At CyberTravels.** A platform whose members were agents shipped its database key to every browser and left row-level security off. CyberTravels' vector store and CRM sit behind the same kind of API.

## 2 · The framework

```
   browser                    Supabase Data API           agents table
   +-----------+              +-----------------+         +---------------+
   | anon key  | -----------> |  row-level      |  ...    | id            |
   | (by       |              |  security       |         | owner         |
   |  design)  |              |  DISABLED       | ------> | provider_key  |  <-
   +-----------+              +-----------------+         | claim_token   |
                                                          +---------------+
   the anon key was never the problem. the absent policy was.

   what leaked            who can revoke it
   session token          Moltbook
   claim token            Moltbook
   provider API key       the person who created the agent   <- not the platform
```

Moltbook launched in late January 2026 as a social network with an unusual
membership rule: the accounts were autonomous AI agents, posting, commenting and
forming communities, with humans watching. Within days a security researcher,
Jameson O'Reilly, found that the whole thing was readable by anyone.

The mechanism is almost disappointingly small. Moltbook ran on **Supabase**, and
the site shipped its Supabase URL and publishable ("anon") key in the client —
which is normal and by design. What was not normal is that **Row-Level Security
was disabled on the tables behind it**. In Supabase, RLS is what turns "this key
identifies the application" into "this key may read this row". Without it, the
anon key is a read-everything key.

So a single query returned every agent's record. Those records held, in
plaintext, in a client-readable table:

- each agent's **secret API key** — spanning OpenAI, Anthropic, AWS, GitHub and
  Google Cloud accounts belonging to the humans who created them,
- claim tokens and verification codes,
- the owner relationships linking every agent back to its creator.

Anyone holding those keys could impersonate any agent on the platform, post as
it, and drive it — without ever failing an authentication check, because they
were authenticating correctly, as the agent.

Two things make this a Function C case study rather than a footnote.

**The blast radius is not the platform.** A social network for agents losing its
own data is a bad day. A social network for agents losing the *provider
credentials of everyone who registered one* is an incident in every one of those
providers' accounts, and the platform cannot revoke them for you.

**The second surface was never touched.** Moltbook's architecture — agents
ingesting and acting on content other agents post — is an indirect prompt
injection surface by construction (A1.3). The breach did not use it. It did not
need to.

The fix was two SQL statements.

> **Sources.** Public reporting on the Moltbook disclosure, late January 2026:
> [Treblle's breakdown](https://treblle.com/blog/moltbook-breach-breakdown),
> [PointGuard AI](https://www.pointguardai.com/ai-security-incidents/moltbook-ai-agent-network-platform-vulnerability),
> [Vectra AI](https://www.vectra.ai/blog/moltbook-and-the-illusion-of-harmless-ai-agent-communities)
> and [Kiteworks](https://www.kiteworks.com/cybersecurity-risk-management/moltbook-ai-agent-security-threat-enterprise-data-protection/).
> Figures below are theirs. Reporting disagrees on the scale — 770,000 agents
> in one account, 1.5 million in another — and both are carried here rather
> than one being chosen.

<svg viewBox="0 0 700 168" width="100%" style="max-width:700px;height:auto;font-family:ui-monospace,SFMono-Regular,Menlo,monospace;font-size:12px"><defs><marker id="a" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="6" markerHeight="6" orient="auto-start-reverse"><path d="M 0 0 L 10 5 L 0 10 z" fill="#8A93A6"/></marker></defs><rect x="6" y="16" width="190" height="62" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="101.0" y="44.0" text-anchor="middle" fill="currentColor">browser</text><text x="101.0" y="60.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">anyone, unauthenticated</text><text x="101" y="96" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">ships the Supabase URL</text><text x="101" y="110" text-anchor="middle" fill="#8A93A6" font-size="11" font-weight="normal">+ the anon key (by design)</text><rect x="268" y="16" width="170" height="62" rx="5" fill="none" stroke="#E0912F" stroke-width="1.4"/><text x="353.0" y="51.0" text-anchor="middle" fill="#E0912F">Supabase Data API</text><rect x="510" y="6" width="184" height="40" rx="5" fill="none" stroke="#E05C4B" stroke-width="1.4" stroke-dasharray="5 4"/><text x="602.0" y="30.0" text-anchor="middle" fill="#E05C4B">row-level security</text><text x="602" y="60" text-anchor="middle" fill="#E05C4B" font-size="11" font-weight="600">DISABLED</text><rect x="510" y="78" width="184" height="52" rx="5" fill="none" stroke="currentColor" stroke-width="1.4"/><text x="602.0" y="101.0" text-anchor="middle" fill="currentColor">agents table</text><text x="602.0" y="117.0" text-anchor="middle" fill="#8A93A6" font-size="10.5">every row, every column</text><line x1="197" y1="47" x2="265" y2="47" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><line x1="439" y1="47" x2="506" y2="100" stroke="#8A93A6" stroke-width="1.4" marker-end="url(#a)"/><text x="350" y="150" text-anchor="middle" fill="#8A93A6" font-size="11.5" font-weight="normal">the anon key was never the problem. the absent policy was.</text></svg><div style="font-size:12px;color:#8A93A6;margin-top:2px">RLS is what turns 'this key identifies the application' into 'this key may read this row'. Without it the anon key reads everything.</div>

## 3 · The query, and the two statements that close it

This is worth running rather than drawing, because the interesting part is what comes back — and how little has to change for it to stop.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)"></th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">SQL</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">1</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><code>alter table agents enable row level security;</code></td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">2</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><code>create policy owner_reads on agents for select using (auth.uid() = owner_id);</code></td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">Reported as roughly two statements. The gap between an incident and no incident was a policy nobody wrote, not a control nobody could afford.</div>

## 4 · Why the blast radius is not the platform's

Moltbook losing its own data would be a bad day for Moltbook. What was in the table belonged to everyone who had registered an agent, and Moltbook could not revoke any of it.

## 5 · The surface the attack did not need

Worth saying plainly, because it is the part that generalises: the interesting architectural risk in Moltbook was never exercised.

<table style="border-collapse:collapse;margin:4px 0 2px;width:100%"><thead><tr><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">surface</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">status in this incident</th><th style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px;font-weight:600;border-bottom:2px solid rgba(138,147,166,.7)">where it is taught</th></tr></thead><tbody><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">credential store readable by anyone</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px"><b>used — this was the breach</b></td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A3.8, and the Supabase pattern in C2.10</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">agents ingest and act on other agents&#x27; posts</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">present, untouched</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A1.3 indirect prompt injection, A1.10 comms poisoning</td></tr><tr><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">agents coordinating at population scale</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">present, untouched</td><td style="padding:6px 11px;border-bottom:1px solid rgba(138,147,166,.35);text-align:left;vertical-align:top;font-size:13px">A1.11, D1.10 fleet correlation</td></tr></tbody></table><div style="font-size:12px;color:#8A93A6;margin-top:6px">An architecture can hold two novel risks and still be undone by a missing row policy. Novelty is not the same as likelihood.</div>

## 6 · The procedure, as a skill

The publishable key is meant to ship in a client; the rows are not. The skill queries every table anonymously with row-level policy off and on, and counts the provider credentials a leaked key returns — along with who can revoke them.

In [ ]:
# skills/research/row-level-policy-check/SKILL.md — embedded verbatim from the repository.
# This is the file itself, not a paraphrase of it.
SKILL_MD = r"""---
name: row-level-policy-check
description: >-
  Query an exposed data API as an anonymous caller with row-level security off
  and on, and count what a publishable key returns. Use when a client-side key
  reaches a database directly, or when a platform's default is open until
  somebody closes it.
allowed-tools: Read, Grep, Glob
---

# The anonymous key is meant to be public; the rows are not

Platforms that expose a database over HTTP hand out a key designed to ship in a
client. That is safe exactly when row-level policy is on, and the default on
several of them is off. The check is one query, run twice, and it is the
difference between a key that is public by design and a table that is.

## When to use this

Any deployment where a browser or an agent talks to a database service directly,
and any platform whose quickstart hands you an anonymous key.

## Procedure

**1 — Get the publishable key from where it actually is.** The client bundle,
the mobile app, the agent's configuration. It is not a secret and treating it as
one is what hides this defect.

**2 — Enumerate the tables.** Use the platform's catalogue rather than the
application's own queries — the application only touches the tables it needs,
which is not the set that exists.

**3 — Query each table anonymously with policy off.** Record the row count and,
specifically, whether any row contains a credential: provider keys stored
alongside agent configuration are the common and expensive case.

**4 — Enable policy and re-run.** Anonymous should return nothing; a signed-in
owner should return their own rows and no more. Both halves — a policy that
returns nothing to everyone is an outage.

**5 — Cost the exposure.** For every credential returned, name the provider and
who can revoke it. The revocation owner is usually not you, and that is the
sentence that gets the work scheduled.

## Output contract

```json
{
  "key": {"kind": "publishable", "found_in": "str"},
  "tables": [{"name": "str", "sensitive": true,
              "anon_rows_policy_off": 0, "anon_rows_policy_on": 0, "owner_rows": 0}],
  "credentials_exposed": [{"provider": "str", "revocable_by": "str"}],
  "verdict": "open|closed"
}
```

## Failure modes

- **Testing with the service key.** It bypasses policy; that proves nothing.
- **Enumerating from the application.** It knows about its own tables only.
- **Reporting rows without the credentials.** The provider keys are the
  incident.
"""

In [ ]:
import json, re

def parse_skill(md):
    """Split a SKILL.md into (frontmatter dict, body).

    Frontmatter is a small, fixed subset of YAML: `key: value`, plus folded
    scalars (`description: >-`) whose continuation lines are indented. That is
    all a skill needs, and parsing it directly means no dependency.
    """
    if not md.startswith("---"):
        raise ValueError("a SKILL.md must open with a frontmatter block")
    _, front, body = md.split("---", 2)
    meta, key = {}, None
    for line in front.strip().splitlines():
        if not line.strip():
            continue
        if not line[0].isspace() and ":" in line:
            key, val = line.split(":", 1)
            key, val = key.strip(), val.strip()
            # `>-` and `|` open a folded block; the value is on the next lines
            meta[key] = "" if val in (">-", ">", "|", "|-") else val
        elif key is not None:
            meta[key] = (meta[key] + " " + line.strip()).strip()
    if "allowed-tools" in meta:
        meta["allowed-tools"] = [t.strip() for t in meta["allowed-tools"].split(",")
                                 if t.strip()]
    for required in ("name", "description"):
        if not meta.get(required):
            raise ValueError(f"skill is missing a {required!r}")
    return meta, body.strip()

_WORD = re.compile(r"[a-z][a-z-]{3,}")

def route(task, skills):
    """Pick the skill whose description best matches a task. Deterministic.

    The description is not documentation — it is the routing key. An agent
    decides whether to load a skill by reading it, so a vague description means
    the skill never fires when it should, and two overlapping descriptions mean
    the wrong one fires.

    Returns (pick, scores, margin). A margin of 0 means the top two scored the
    same and the "winner" is just whichever sorted first — an arbitrary answer
    wearing a confident face. Callers should refuse to auto-route on margin 0
    rather than pretend the tiebreak meant something.
    """
    want = set(_WORD.findall(task.lower()))
    def score(meta):
        return len(want & set(_WORD.findall(meta["description"].lower())))
    scores = {n: score(skills[n]) for n in sorted(skills)}
    # sort names first, then by score: ties must break identically on every
    # machine or the same task routes differently on two runs
    ranked = sorted(sorted(skills), key=lambda n: -scores[n])
    top = scores[ranked[0]]
    margin = top - (scores[ranked[1]] if len(ranked) > 1 else 0)
    return ranked[0], scores, margin

def contract_of(body):
    """The JSON block under '## Output contract' — the skill's machine promise."""
    # non-greedy across any prose between the heading and the fence
    m = re.search(r"## Output contract\b.*?```json\n(.*?)```", body, re.S)
    if not m:
        raise ValueError("skill declares no output contract")
    return json.loads(m.group(1))

def check(instance, contract, path="$"):
    """Structural conformance of an instance against a contract template.

    Returns the list of problems. An empty list means the shape is right — and
    that is *all* it means. Conformance is not accuracy: an empty findings list
    conforms perfectly and tells you nothing.
    """
    problems = []
    if isinstance(contract, dict):
        if not isinstance(instance, dict):
            return [f"{path}: expected an object, got {type(instance).__name__}"]
        for k, v in sorted(contract.items()):
            if k not in instance:
                problems.append(f"{path}.{k}: missing")
            else:
                problems += check(instance[k], v, f"{path}.{k}")
    elif isinstance(contract, list):
        if not isinstance(instance, list):
            return [f"{path}: expected a list, got {type(instance).__name__}"]
        for i, item in enumerate(instance):          # every element, same template
            problems += check(item, contract[0], f"{path}[{i}]")
    elif isinstance(contract, str) and "|" in contract:
        if instance not in contract.split("|"):
            problems.append(f"{path}: {instance!r} is not one of {contract}")
    elif isinstance(contract, bool):                  # before the numeric case:
        if not isinstance(instance, bool):            # bool is a subclass of int
            problems.append(f"{path}: expected bool, got {type(instance).__name__}")
    elif isinstance(contract, (int, float)):
        # JSON has one number type. A contract written `0` must accept 0.4, or
        # every cost and rate in the pipeline has to be rounded to satisfy a
        # checker rather than to be correct.
        if isinstance(instance, bool) or not isinstance(instance, (int, float)):
            problems.append(f"{path}: expected a number, got {type(instance).__name__}")
    elif not isinstance(instance, type(contract)):
        problems.append(f"{path}: expected {type(contract).__name__}, "
                        f"got {type(instance).__name__}")
    return problems

# Execute the skill above: parse skills/research/row-level-policy-check/SKILL.md into the two
# halves an agent uses — the frontmatter it routes on, and the body
# it follows.
meta, body = parse_skill(SKILL_MD)
print(f"loaded skill: {meta['name']}")
print(f"  tools it may use: {', '.join(meta.get('allowed-tools', [])) or '—'}")
print(f"  routing description: {len(meta['description'].split())} words")
print(f"  procedure: {len(body.splitlines())} lines")

In [ ]:
# skills/research/row-level-policy-check/scripts/row_level_policy_check.py — embedded verbatim from the repository.
# This is the skill's own script, not a paraphrase of it.
#!/usr/bin/env python3
"""Query an exposed data API as an anonymous caller with row-level security off and on, and count what a leaked key returns.

This is the executable half of the `row-level-policy-check` skill: the check the
SKILL.md next to it describes, run against a synthetic CyberTravels
estate so two runs can be diffed and the result argued with.

Standard library only, and deterministic, so it runs on a Kaggle
kernel with the internet switched off.
"""

AGENTS = [
 {"id": "a-0001", "owner": "dana@example",  "handle": "@researchbot",
  "provider_key": "sk-REDACTED-openai",     "claim_token": "clm_8fA2"},
 {"id": "a-0002", "owner": "sam@example",   "handle": "@newsdigest",
  "provider_key": "sk-ant-REDACTED",        "claim_token": "clm_2bQ7"},
 {"id": "a-0003", "owner": "kim@example",   "handle": "@dealfinder",
  "provider_key": "AKIA-REDACTED-aws",      "claim_token": "clm_9zR1"},
]

def data_api(table, caller, rls_enabled):
    """Supabase's PostgREST surface, in miniature.

    `caller` is whoever the anon key resolves to - which is nobody in
    particular. With RLS off there is no policy to consult, so every row is
    returned; with RLS on, the policy decides.
    """
    if not rls_enabled:
        return list(table)                       # no policy exists to consult
    return [r for r in table if r["owner"] == caller]

anon = None                                      # the anon key is not a person
for label, rls in (("RLS disabled (as shipped)", False), ("RLS enabled", True)):
    rows = data_api(AGENTS, caller=anon, rls_enabled=rls)
    print(f"{label:28s}rows returned: {len(rows)}")
    for r in rows:
        print(f"      {r['handle']:14s}{r['owner']:16s}{r['provider_key']}")

owner_rows = data_api(AGENTS, caller="dana@example", rls_enabled=True)
print(f"\nsigned in as dana@example, RLS enabled: {len(owner_rows)} row")
print()
print("Same key, same endpoint, same table. The only difference is whether a")
print("policy exists for the API to consult.")
assert len(data_api(AGENTS, anon, False)) == 3
assert len(data_api(AGENTS, anon, True)) == 0 and len(owner_rows) == 1

REPORTED_SCALE = {"Treblle": 770_000, "Wiz-sourced reporting": 1_500_000}
PROVIDERS = ["OpenAI", "Anthropic", "AWS", "GitHub", "Google Cloud"]

for source, n in sorted(REPORTED_SCALE.items()):
    print(f"{source:26s}{n:>10,} agents exposed")
low, high = min(REPORTED_SCALE.values()), max(REPORTED_SCALE.values())
print(f"\nreported range            {low:>10,} - {high:,}")
print(f"provider accounts implicated: {', '.join(PROVIDERS)}")

# Who can actually revoke each thing that leaked.
REVOCABLE_BY = {
 "the Moltbook session token": "Moltbook",
 "the claim token":            "Moltbook",
 "the agent's provider key":   "the individual who created the agent",
}
print()
print(f"{'what leaked':30s}who can revoke it")
for what in sorted(REVOCABLE_BY):
    print(f"{what:30s}{REVOCABLE_BY[what]}")

platform_can_fix = [w for w in REVOCABLE_BY if REVOCABLE_BY[w] == "Moltbook"]
print(f"\nthe platform can revoke {len(platform_can_fix)} of {len(REVOCABLE_BY)}.")
print("The third is a key in somebody else's provider account, and the only")
print("person who can turn it off may not know it was ever exposed. That is the")
print("difference between a platform breach and a supply-chain one.")
assert len(platform_can_fix) == 2

## What you just proved

With RLS disabled the anon key returns all three agent rows, secret provider keys included; with RLS enabled and no signed-in user it returns none, and one row for the owner. Reported scale spans 770,000 to 1.5 million agents across five providers, and of the three things that leaked the platform can revoke two — the third is a key in somebody else's account.

## Your turn

Run `select relname from pg_class where relrowsecurity = false` against your own project, or the equivalent for whatever backend you use. Then find the table holding anything credential-shaped and check whether it is reachable from the client at all — the answer to the second question is the one that decides the size of your bad day.

---

**Next → [C2.10 · Case study — the Supabase pattern: open until closed](https://spbreed.github.io/cyber-commons/lessons/C2.10.html)**

[All lessons](https://spbreed.github.io/cyber-commons/lessons/) · [This lesson's page](https://spbreed.github.io/cyber-commons/lessons/C2.9.html) · [Source](https://github.com/spbreed/cyber-commons/blob/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks/C2.9.ipynb)

*Cyber Commons — a free, open commons for Cyber AI.*